# NC-SSM TASLP: Full Experiment Pipeline
**NC-SSM 7.4K | NC-SSM-Large 10.2K | NC-SSM+NanoSE 10.5K | BC-ResNet-1 7.5K | NM-Matched 7.4K | DS-CNN-S 23.8K**

---

## Execution Order

| Step | Description | Time |
|------|-------------|------|
| **0** | Setup (GPU + Drive) | 30s |
| **1** | Clone + Data + Checkpoints | ~3min |
| **2** | Train ALL Paper Models (NC-SSM, NC-SSM-L, BC-ResNet-1, NM-Matched, DS-CNN-S) | ~50min |
| **2e** | Train NanoSE (NC-SSM + Enhancement Expert) | ~15min |
| **3** | Full 7-SNR Noise Evaluation (now -25 to 15 dB) | ~20min |
| **4** | SS v2 vs SS v3 Comparison | ~30min |
| **5** | SS+Bypass Optimization Sweep | ~30min |
| **6** | SM-SSM + SAGN Training (ablation) | ~20min |
| **7** | Full 6-Model Comparison | ~20min |
| **8** | Sigma Gate Analysis | ~5min |
| **9** | **NASG Training + Eval** (low-SNR improvement) | ~25min |

## Step 0: Setup

In [1]:
# GPU check
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠ No GPU! Go to: Runtime > Change runtime type > GPU')

# Mount Drive (for checkpoint persistence)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

GPU: Tesla T4
Memory: 15.6 GB
Mounted at /content/drive


## Step 1: Clone + Data + Recovery
**세션 끊어져도 Drive에서 체크포인트 자동 복원**
- 기존 `NanoMamba/checkpoints_full`의 baseline 체크포인트도 복원
- 누락 모델 자동 감지

In [ ]:
%%time
import os, shutil, torch

# ============================================================
# 1. Clone code
# ============================================================
os.chdir('/content')  # Restore CWD (may point to deleted dir from previous run)
!rm -rf /content/NC-SSM-TASLP
!git clone https://github.com/DrJinHoChoi/NC-SSM-TASLP.git /content/NC-SSM-TASLP
os.chdir('/content/NC-SSM-TASLP')

# ============================================================
# 2. Download GSC V2
# ============================================================
if not os.path.exists('./data/speech_commands_v0.02'):
    !mkdir -p data && cd data && \
     wget -q http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz && \
     mkdir -p speech_commands_v0.02 && \
     tar -xzf speech_commands_v0.02.tar.gz -C speech_commands_v0.02 && \
     rm speech_commands_v0.02.tar.gz
    print('GSC V2 downloaded')
else:
    print('GSC V2 already exists')

# ============================================================
# 3. Recovery: Restore checkpoints from Drive
# ============================================================
DRIVE_NEW = '/content/drive/MyDrive/NC-SSM-TASLP/checkpoints_full'
DRIVE_OLD = '/content/drive/MyDrive/NanoMamba/checkpoints_full'
LOCAL_CKPT = './checkpoints_full'
os.makedirs(LOCAL_CKPT, exist_ok=True)

# Required models for full experiment
REQUIRED = {
    'NanoMamba-NC-Large':  'NC-SSM-Large (10.2K) — 최종 제안 모델',
    'NanoMamba-NC-Matched': 'NC-SSM (7.4K) — 기본 제안 모델',
    'BC-ResNet-1':          'BC-ResNet-1 (7.5K) — CNN baseline',
    'NanoMamba-Matched-DualPCEN-v2-SSMv2': 'NM-Matched (7.4K) — SSM baseline',
    'DS-CNN-S':             'DS-CNN-S (23.8K) — Large CNN',
    'NanoMamba-Matched-SM': 'SM-SSM (7.4K) — ablation',
    'SAGN':                 'SAGN (7.2K) — CNN+LSG ablation',
}

def restore_checkpoint(model_name):
    """Try to restore checkpoint from Drive (new path first, then old path)."""
    local_dir = os.path.join(LOCAL_CKPT, model_name)
    local_best = os.path.join(local_dir, 'best.pt')
    if os.path.exists(local_best):
        return True  # Already local

    # Try new Drive path first
    for drive_path in [DRIVE_NEW, DRIVE_OLD]:
        src = os.path.join(drive_path, model_name, 'best.pt')
        if os.path.exists(src):
            os.makedirs(local_dir, exist_ok=True)
            shutil.copy2(src, local_best)
            return True
    return False

print('='*60)
print('CHECKPOINT RECOVERY')
print('='*60)
found, missing = [], []
for name, desc in REQUIRED.items():
    if restore_checkpoint(name):
        ckpt = torch.load(os.path.join(LOCAL_CKPT, name, 'best.pt'),
                          map_location='cpu')
        acc = ckpt.get('val_acc', 'N/A')
        print(f'  ✓ {name}: val_acc={acc}')
        found.append(name)
    else:
        print(f'  ✗ {name}: NOT FOUND — needs training')
        missing.append(name)

print(f'\n  Recovered: {len(found)}/{len(REQUIRED)}')
if missing:
    print(f'  Missing:   {", ".join(missing)}')
    print(f'  → Run training steps for missing models')
else:
    print(f'  All checkpoints ready! Skip to Step 3 (eval)')

# ============================================================
# 4. Helper: save_to_drive() — call after any training
# ============================================================
def save_to_drive():
    """Sync all local checkpoints to Drive for persistence."""
    os.makedirs(DRIVE_NEW, exist_ok=True)
    for d in os.listdir(LOCAL_CKPT):
        src = os.path.join(LOCAL_CKPT, d, 'best.pt')
        if os.path.exists(src):
            dst_dir = os.path.join(DRIVE_NEW, d)
            os.makedirs(dst_dir, exist_ok=True)
            shutil.copy2(src, os.path.join(dst_dir, 'best.pt'))
    print(f'✓ Checkpoints synced to Drive ({DRIVE_NEW})')

## Step 2: Train ALL Paper Models

### 2a: NC-SSM (7,443 params)
- `d_model=20, d_state=6, n_layers=2`
- Target: ≥95.3% clean accuracy

### 2b: NC-SSM-Large (~10,191 params)
- `d_model=24, d_state=8, n_layers=2`
- `use_tiny_conv` 제거됨 (per-sub-band SNR estimation 교란 방지)
- Target: ≥95% clean accuracy

In [ ]:
import os, shutil

# ============================================================
# 2a: Train NC-SSM (7,443 params)
# ============================================================
print('='*60)
print('2a: Training NC-SSM (d_model=20, d_state=6, 7,443 params)')
print('='*60)
!python train_colab.py \
    --models NanoMamba-NC-Matched \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --noise_aug --noise_curriculum_v2 \
    --calibrate

save_to_drive()
print('\n✓ NC-SSM saved to Drive')

# ============================================================
# 2b: Train NC-SSM-Large (~10,191 params, TinyConv 제거)
# ============================================================
# 기존 NC-SSM-Large 체크포인트 삭제 (구조 변경됨)
old_ckpt = './checkpoints_full/NanoMamba-NC-Large'
if os.path.exists(old_ckpt):
    shutil.rmtree(old_ckpt)
    print(f'✗ Deleted old NC-SSM-Large checkpoint')
drive_ckpt = '/content/drive/MyDrive/NC-SSM-TASLP/checkpoints_full/NanoMamba-NC-Large'
if os.path.exists(drive_ckpt):
    shutil.rmtree(drive_ckpt)

print('\n' + '='*60)
print('2b: Training NC-SSM-Large (d_model=24, d_state=8, ~10,191 params)')
print('='*60)
!python train_colab.py \
    --models NanoMamba-NC-Large \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --noise_aug --noise_curriculum_v2 \
    --calibrate

save_to_drive()
print('\n✓ NC-SSM-Large saved to Drive')
print('\n' + '='*60)
print('BOTH MODELS TRAINED AND SAVED TO DRIVE')
print('='*60)

### 2c: BC-ResNet-1 (7,464 params)
- CNN baseline: Sub-Spectral Norm + Broadcasted Residual
- `scale=1` (tau=1), 동일 조건 (30 epochs, noise-aug, no SA/EMA)

In [ ]:
# ============================================================
# 2c: Train BC-ResNet-1 (7,464 params)
#     CNN baseline — Sub-Spectral Norm + Broadcasted Residual
#     동일 조건: 30 epochs, noise-aug + curriculum v2, no SA/EMA
# ============================================================
import os
print('='*60)
print('2c: Training BC-ResNet-1 (scale=1, 7,464 params)')
print('    Sub-Spectral Norm: per-sub-band BN (frequency-aware)')
print('    Broadcasted Residual: freq pooling + broadcast')
print('='*60)

!python train_colab.py \
    --models BC-ResNet-1 \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --noise_aug --noise_curriculum_v2

save_to_drive()
print('\n✓ BC-ResNet-1 saved to Drive')

### 2d: NM-Matched (7,402 params) + DS-CNN-S (23,756 params)
- **NM-Matched**: SSM baseline (DualPCEN v2 + SA-SSM v2, NC 컴포넌트 없음)
- **DS-CNN-S**: Large CNN baseline (3.2x more params)
- 논문 Table 3, 4에 필요한 전체 baseline 모델

In [ ]:
# ============================================================
# 2d-1: Train NM-Matched (7,402 params)
#        SSM baseline — DualPCEN v2 + SA-SSM v2 (no NC components)
#        논문 Table 3: 95.1% clean, Table 4 baseline
# ============================================================
import os
print('='*60)
print('2d-1: Training NM-Matched (7,402 params)')
print('      SSM baseline: DualPCEN v2 + SA-SSM v2')
print('='*60)

!python train_colab.py \
    --models NanoMamba-Matched-DualPCEN-v2-SSMv2 \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --noise_aug --noise_curriculum_v2

save_to_drive()
print('\n✓ NM-Matched saved to Drive')

# ============================================================
# 2d-2: Train DS-CNN-S (23,756 params)
#        Large CNN baseline — 3.2x more params than BC-ResNet-1
#        논문 Table 3: 96.4% clean, Table 4 reference
# ============================================================
print('\n' + '='*60)
print('2d-2: Training DS-CNN-S (23,756 params)')
print('      Large CNN baseline (3.2x BC-ResNet-1 params)')
print('='*60)

!python train_colab.py \
    --models DS-CNN-S \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --noise_aug --noise_curriculum_v2

save_to_drive()
print('\n✓ DS-CNN-S saved to Drive')
print('\n' + '='*60)
print('ALL BASELINES TRAINED: NM-Matched + DS-CNN-S')
print('='*60)

### 2e: NanoSE — Nano Speech Enhancement Expert (~10,457 params)
- **Sequential Enhancement Expert**: NanoSE (mel IRM) → DualPCEN v2 → NC-SSM classifier
- NanoSE: 257 params (sub-band grouped causal conv + SNR FiLM + bypass gate)
- Based on NC-SSM-Large (10,200) + NanoSE (257) = ~10,457 params
- At high SNR: bypass gate ≈ 1 (no artifact). At low SNR: IRM mask applied.

In [ ]:
# ============================================================
# 2e: Train NanoSE — Nano Speech Enhancement Expert
#     NC-SSM-Large + NanoSE (257 params) = ~10,457 params
#     Sequential: NanoSE (mel IRM) → DualPCEN v2 → NC-SSM
# ============================================================
import os
print('='*60)
print('2e: Training NC-SSM + NanoSE (~10,457 params)')
print('    NanoSE: sub-band grouped causal conv (257 params)')
print('    Sequential: NanoSE → DualPCEN v2 → NC-SSM classifier')
print('='*60)

!python train_colab.py \n    --models NanoMamba-NC-NanoSE \n    --data_dir ./data \n    --checkpoint_dir ./checkpoints_full \n    --epochs 30 \n    --noise_aug --noise_curriculum_v2

save_to_drive()
print('
✓ NC-SSM + NanoSE saved to Drive')

# ============================================================
# 2e-2: NanoSE with HARD 5 epochs (compare with default 10)
# ============================================================
print('
' + '='*60)
print('2e-2: Training NC-SSM + NanoSE (HARD 5 epochs)')
print('      NanoSE handles cleaning → less HARD training needed')
print('='*60)

import shutil
nanose_ckpt = './checkpoints_full/NanoMamba-NC-NanoSE'
nanose_h5_ckpt = './checkpoints_full/NanoMamba-NC-NanoSE-H5'

!python train_colab.py \n    --models NanoMamba-NC-NanoSE \n    --data_dir ./data \n    --checkpoint_dir ./checkpoints_full \n    --epochs 30 \n    --noise_aug --noise_curriculum_v2 \n    --hard_epochs 5

# Rename checkpoint for H5 variant
if os.path.exists(nanose_ckpt):
    if os.path.exists(nanose_h5_ckpt):
        shutil.rmtree(nanose_h5_ckpt)
    shutil.copytree(nanose_ckpt, nanose_h5_ckpt)
    print(f'✓ Copied NanoSE checkpoint to {nanose_h5_ckpt}')

save_to_drive()
print('
✓ NanoSE (HARD 5) saved to Drive')


## Step 3: Full 7-SNR Noise Evaluation
NC-SSM-Large vs NC-SSM vs BC-ResNet-1

In [ ]:
# Full noise evaluation: 5 noise x 7 SNR = 35 conditions
!python train_colab.py \
    --models NanoMamba-NC-Large,NanoMamba-NC-Matched,BC-ResNet-1 \
    --eval_only \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15 \
    --calibrate --calibrate_continuous

## Step 4: SS+Bypass Evaluation (v2 + v3 비교)
- **SS v2**: Subtractive, SF-weighted, oversubtract [0.5, 3.0]
- **SS v3** (NEW): Wiener gain, aggressive, oversubtract [1.0, 5.0], no SF weight

In [ ]:
# ============================================================
# SS v2 (기존): NC-SSM + NC-SSM-Large + BC-ResNet-1
# ============================================================
print('='*70)
print('SS v2: Subtractive + SF-weighted + Bypass v2')
print('='*70)
!python train_colab.py \
    --models NanoMamba-NC-Large,NanoMamba-NC-Matched,BC-ResNet-1 \
    --eval_only \
    --use_enhancer --enhancer_bypass \
    --ss_version v2 --bypass_version v2 \
    --bypass_threshold -2.0 --bypass_scale 3.0 --sf_range 2.0 \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15 \
    --calibrate --calibrate_continuous

# ============================================================
# SS v3 (NEW): NC-SSM + NC-SSM-Large + BC-ResNet-1
# Wiener gain + aggressive alpha + no SF weight
# ============================================================
print('\n' + '='*70)
print('SS v3: Wiener gain + aggressive (alpha 1.0-5.0) + low floor')
print('='*70)
!python train_colab.py \
    --models NanoMamba-NC-Large,NanoMamba-NC-Matched,BC-ResNet-1 \
    --eval_only \
    --use_enhancer --enhancer_bypass \
    --ss_version v3 --bypass_version v2 \
    --bypass_threshold -2.0 --bypass_scale 3.0 --sf_range 2.0 \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15 \
    --calibrate --calibrate_continuous

print('\n' + '='*70)
print('COMPARE: SS v2 vs v3 at -15dB (improvement) and 0dB (degradation)')
print('='*70)

## Step 5: SS+Bypass Optimization Sweep
**v4 파라미터가 최적인지 확인**: sf_range, bypass_threshold, bypass_scale

In [ ]:
# ============================================================
# SS+Bypass Parameter Sweep: Is v4 (sf_range=2.0) optimal?
# Test 4 configurations and compare at -15dB and 0dB
# ============================================================

configs = [
    # (name, sf_range, threshold, scale)
    ('v4-current',  2.0, -2.0, 3.0),  # Current default
    ('v4-tight',    1.0, -1.0, 4.0),  # Tighter bypass: more aggressive at 0dB
    ('v4-wide',     3.0, -3.0, 2.0),  # Wider range: more SS at moderate SNR
    ('v4-noadapt',  0.0, -2.0, 3.0),  # No SF adaptation: same threshold all noises
]

print('='*70)
print('SS+BYPASS v4 PARAMETER SWEEP')
print('='*70)

for name, sf_range, threshold, scale in configs:
    print(f'\n{"="*70}')
    print(f'Config: {name} (sf_range={sf_range}, threshold={threshold}, scale={scale})')
    print(f'{"="*70}')
    !python train_colab.py \
        --models NanoMamba-NC-Large \
        --eval_only \
        --use_enhancer --enhancer_bypass \
        --ss_version v2 --bypass_version v2 \
        --bypass_threshold {threshold} --bypass_scale {scale} --sf_range {sf_range} \
        --data_dir ./data \
        --checkpoint_dir ./checkpoints_full \
        --noise_types factory,white,babble,street,pink \
        --snr_range=-15,0 \
        --calibrate --calibrate_continuous

print('\n' + '='*70)
print('SWEEP COMPLETE: Compare -15dB improvement vs 0dB degradation')
print('Optimal config should maximize -15dB gain while minimizing 0dB loss')
print('='*70)

## Step 6: SM-SSM + SAGN Training (TASLP ablation baselines)
논문 스토리: NM-Matched → SM-SSM → NC-SSM 진화

In [ ]:
# Train SM-SSM (intermediate step: +38 params for sigma gate)
!python train_colab.py \
    --models NanoMamba-Matched-SM \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --noise_aug --noise_curriculum_v2 \
    --calibrate

save_to_drive()  # Recovery point after SM-SSM

# Train SAGN (CNN+LSG baseline: proves LSG alone on CNN < NC-SSM)
!python train_colab.py \
    --models SAGN \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --noise_aug --noise_curriculum_v2 \
    --calibrate

save_to_drive()  # Recovery point after SAGN

## Step 7: Full 6-Model TASLP Comparison
**All models × 5 noise × 7 SNR = complete Table V**

In [ ]:
# TASLP Table V: Complete 6-model noise evaluation
!python train_colab.py \
    --models NanoMamba-NC-Large,NanoMamba-NC-Matched,NanoMamba-Matched-SM,SAGN,BC-ResNet-1,NanoMamba-Matched-DualPCEN-v2-SSMv2,DS-CNN-S \
    --eval_only \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15 \
    --calibrate --calibrate_continuous

# TASLP Table VII: SS+Bypass for all models
!python train_colab.py \
    --models NanoMamba-NC-Large,NanoMamba-NC-Matched,NanoMamba-Matched-SM,NanoMamba-Matched-DualPCEN-v2-SSMv2,BC-ResNet-1,DS-CNN-S,SAGN \
    --eval_only \
    --use_enhancer --enhancer_bypass \
    --ss_version v2 --bypass_version v2 \
    --bypass_threshold -2.0 --bypass_scale 3.0 --sf_range 2.0 \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15 \
    --calibrate --calibrate_continuous

print('\n' + '='*60)
print('TASLP FULL COMPARISON COMPLETE')
print('='*60)

## Step 8: Sigma Gate Analysis (Fig. 3 data)
Per-sub-band selectivity visualization data

In [ ]:
# Sigma gate per-sub-band analysis for NC-SSM-Large
import torch, sys, os
sys.path.insert(0, '/content/NC-SSM-TASLP')
from train_colab import (
    SpeechCommandsDataset, analyze_selectivity_gates,
    generate_noise_signal, mix_audio_at_snr
)
from nanomamba import create_nanomamba_nc_large

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load NC-SSM-Large checkpoint
model = create_nanomamba_nc_large(n_classes=12).to(device)
ckpt_path = './checkpoints_full/NanoMamba-NC-Large/best.pt'
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded NC-SSM-Large: val_acc={ckpt.get("val_acc", "N/A")}')
else:
    print(f'Checkpoint not found: {ckpt_path}')
    print('Run Step 2 first!')

model.eval()

# Load dataset for analysis
val_dataset = SpeechCommandsDataset('./data/speech_commands_v0.02', split='validation')

# Analyze sigma gates
analyze_selectivity_gates(model, val_dataset, device=device)

## Step 9: NASG Training — Noise-Aware Selective Gating
**저 SNR에서 CNN upper bound를 넘는 구조**

| Component | Description | Params |
|-----------|-------------|--------|
| `nasg_scale` | SNR-dependent gate steepness (init=20) | 1/block |
| `nasg_bias` | Gate threshold (init=-5) | 1/block |
| **Total** | 2 blocks × 2 params | **+4** |

**이론적 원리:**
- Low SNR: `nasg_w → 0` → selective params → 0 → LTI mode → O(σ²)
- High SNR: `nasg_w → 1` → full selectivity preserved
- -15dB에서 effective selectivity: σ·g ≈ 0.001 (2,500× noise reduction)

### 9a: Train NC-SSM-Large-NASG (10,195 params)
### 9b: Compare with baseline at -15dB (target: White 34.1% → 50%+)

In [ ]:
import os, shutil

# ============================================================
# 9a: Train NC-SSM-Large-NASG (10,195 params)
#     Noise-Aware Selective Gating for low-SNR improvement
# ============================================================
print('='*70)
print('9a: Training NC-SSM-Large-NASG (d_model=24, d_state=8, +NASG)')
print('    → nasg_scale=20, nasg_bias=-5 (steep gate for noise suppression)')
print('='*70)

!python train_colab.py \
    --models NanoMamba-NC-Large-NASG \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --noise_aug --noise_curriculum_v2 --low_snr_boost \
    --calibrate

save_to_drive()
print('\n✓ NC-SSM-Large-NASG saved to Drive')

# ============================================================
# 9b: Full noise evaluation — NASG vs Baseline vs CNN
# ============================================================
print('\n' + '='*70)
print('9b: NASG vs Baseline vs CNN — Full noise evaluation')
print('='*70)

!python train_colab.py \
    --models NanoMamba-NC-Large-NASG,NanoMamba-NC-Large,BC-ResNet-1 \
    --eval_only \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15 \
    --calibrate --calibrate_continuous

print('\n' + '='*70)
print('KEY METRICS TO CHECK:')
print('  White  -15dB: Baseline 34.1% → NASG target 50%+')
print('  White  -10dB: Baseline 59.8% → NASG target 65%+')
print('  Clean       : Must stay ≥95.0% (no regression)')
print('  Factory -15dB: Baseline 61.3% → Check improvement')
print('='*70)

## Step 9c: NASG + SS+Bypass Evaluation
**NASG 체크포인트에 SS v2/v3 + Bypass 적용하여 White/Pink 추가 개선 확인**

이론적 배경:
- SS가 input noise를 α배 줄이면 → SSM internal variance α⁶ 감소 (CNN은 α²만)
- NASG가 이미 selectivity를 suppression → SS와 시너지 기대
- White noise at -15dB: 41.6% → SS로 50%+ 목표

In [ ]:
# ============================================================
# 9c: NASG + SS+Bypass (v2 & v3) — White/Pink noise target
# ============================================================

# --- SS v2 + Bypass ---
print('='*70)
print('9c-1: NASG + SS v2 + Bypass')
print('='*70)
!python train_colab.py \
    --models NanoMamba-NC-Large-NASG \
    --eval_only \
    --use_enhancer --enhancer_bypass \
    --ss_version v2 --bypass_version v2 \
    --bypass_threshold -2.0 --bypass_scale 3.0 --sf_range 2.0 \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15

# --- SS v3 (Wiener) + Bypass ---
print('\n' + '='*70)
print('9c-2: NASG + SS v3 (Wiener) + Bypass')
print('='*70)
!python train_colab.py \
    --models NanoMamba-NC-Large-NASG \
    --eval_only \
    --use_enhancer --enhancer_bypass \
    --ss_version v3 --bypass_version v2 \
    --bypass_threshold -2.0 --bypass_scale 3.0 --sf_range 2.0 \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15

print('\n' + '='*70)
print('NASG + SS COMPARISON:')
print('  NASG only    — White -15dB: 41.6%')
print('  NASG + SS v2 — White -15dB: ???')
print('  NASG + SS v3 — White -15dB: ???')
print('  BC-ResNet-1  — White -15dB: 58.6% (target)')
print('='*70)

## Step 10: NASG v2 — State Masking (Input-Gated) + Mild White/Pink Boost
**Bug fixes from v2 failure: Phase 2A removed, Phase 3A fixed, Phase 1B removed**

| Phase | Description | Status |
|-------|-------------|--------|
| 1A | White/Pink 가중 샘플링 (1.3×, mild) | ✅ Reduced from 2.0× |
| ~~1B~~ | ~~Per-sample loss weighting~~ | ❌ Removed (too aggressive) |
| ~~2A~~ | ~~Temporal smoothing~~ | ❌ Removed (counterproductive at -15dB) |
| 3A | Adaptive state masking (**input-gated**, not output) | ✅ Fixed |
| **Total** | NASG 4 + state_mask 4 | **10,199** |

**v2 실패 원인 요약:**
1. State gate가 output에만 적용됨 → 노이즈 축적 방치 → 33.5% (v1의 41.6%에서 퇴행)
2. Phase 2A smoothing이 -15dB에서 노이즈를 DC-like로 변환 → SSM이 추적
3. Phase 1A+1B 2×+2× boost → 학습 불안정 (92%→82%)
4. SS v3 Wiener gain: -15dB에서 모든 bin이 floor로 클리핑 → zero effect

**v2-fix 핵심:** state_gate를 state INPUT에 적용 → 노이즈 유입 자체를 차단

In [ ]:
import os, shutil

# ============================================================
# 10a: Update code (git pull for bug fixes)
#      Fix: origin/main → origin/master (correct branch name)
# ============================================================
print('='*70)
print('10a: Pulling latest code (Phase 3A fix + Phase 2A/1B removal)')
print('='*70)
!cd /content/NC-SSM-TASLP && git pull origin master

# Delete old NASG checkpoint (incompatible: state_gate behavior changed)
old_nasg = './checkpoints_full/NanoMamba-NC-Large-NASG'
if os.path.exists(old_nasg):
    shutil.rmtree(old_nasg)
    print('✗ Deleted old NASG checkpoint (state_gate input-gated now)')
drive_nasg = '/content/drive/MyDrive/NC-SSM-TASLP/checkpoints_full/NanoMamba-NC-Large-NASG'
if os.path.exists(drive_nasg):
    shutil.rmtree(drive_nasg)

# ============================================================
# 10b: Train NASG v2-fix (10,199 params)
#      Fixes: input-gated state_mask, no Phase 2A, mild 1.3× boost
# ============================================================
print('\n' + '='*70)
print('10b: Training NASG v2-fix (Phase 3A input-gated)')
print('    Fixed: state_gate on INPUT (not output)')
print('    Removed: Phase 2A (temporal smoothing), Phase 1B (loss weight)')
print('    Reduced: Phase 1A 2.0× → 1.3× white/pink boost')
print('    Total: 10,199 params')
print('='*70)

!python train_colab.py \
    --models NanoMamba-NC-Large-NASG \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --noise_aug --noise_curriculum_v2 --low_snr_boost \
    --calibrate

save_to_drive()
print('\n✓ NASG v2-fix saved to Drive')

# ============================================================
# 10c: Full eval — NASG v2-fix (with calibration!)
#      Fix: added --calibrate --calibrate_continuous (was missing)
# ============================================================
print('\n' + '='*70)
print('10c: NASG v2-fix full noise evaluation (with calibration)')
print('='*70)

!python train_colab.py \
    --models NanoMamba-NC-Large-NASG \
    --eval_only \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15 \
    --calibrate --calibrate_continuous

# ============================================================
# 10d: NASG v2-fix + SS v2 (not v3!)
#      Fix: SS v3 was zero-effect at -15dB (Wiener floor clips all bins)
#      SS v2 subtractive method provides frequency-differential attenuation
# ============================================================
print('\n' + '='*70)
print('10d: NASG v2-fix + SS v2 + Bypass (calibrated)')
print('='*70)

!python train_colab.py \
    --models NanoMamba-NC-Large-NASG \
    --eval_only \
    --use_enhancer --enhancer_bypass \
    --ss_version v2 --bypass_version v2 \
    --bypass_threshold -2.0 --bypass_scale 3.0 --sf_range 2.0 \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15 \
    --calibrate --calibrate_continuous

print('\n' + '='*70)
print('NASG v2-fix RESULTS COMPARISON:')
print('  Baseline (no NASG)    — White -15dB: 34.1%')
print('  NASG v1 (Step 9)      — White -15dB: 41.6%')
print('  NASG v2 FAILED        — White -15dB: 33.5% (output-gated, Phase 2A harm)')
print('  NASG v2-fix (Step 10) — White -15dB: ???  (target: 43-48%)')
print('  NASG v2-fix + SS v2   — White -15dB: ???  (target: 50-55%)')
print('  BC-ResNet-1           — White -15dB: 58.6%')
print('='*70)

## Step 11: Noise Propagation Diagnostic — CNN vs NC-SSM
**학습 후 노이즈 전파 수치 확인: 의도대로 O(σ²) 달성했는지 검증**

**SNR Teacher-Student 적용 후** (Step 12에서 학습한 모델 사용)

이론적 검증 항목:
| 항목 | 기대값 (White -15dB) | 의미 |
|------|---------------------|------|
| NASG gate g | ≈ 0.011 | Pure LTI mode (σ⁶→σ² 전환) |
| State gate m₀ | ≈ 0.06 | 첫 번째 state 대부분 닫힘 |
| State gate m₇ | ≈ 0.01 | 마지막 state 완전 닫힘 |
| N_eff | ≈ 0.2-1.0 | 유효 state 대폭 축소 |
| Amp.Ratio | ∝ σ² (linear) | O(σ²) 확인: -15dB→-5dB에서 ~10× 감소 |
| ||h||² per state | 높은 state일수록 작음 | Input gating 작동 확인 |

In [ ]:
# ============================================================
# Step 11: Noise Propagation Diagnostic
# Verifies that NASG + State Masking achieve O(sigma^2)
# Self-contained: works after Step 0 + Step 12 (no Step 1 needed)
# ============================================================
import torch, sys, os

REPO_DIR = '/content/NC-SSM-TASLP'
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

from torch.utils.data import DataLoader
from train_colab import (
    SpeechCommandsDataset, diagnose_noise_propagation,
    _create_mel_fb_tensor, BCResNet
)
from nanomamba import create_nanomamba_nc_large

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ============================================================
# 11a: Load NC-SSM-Large-NASG (proposed model)
# ============================================================
model = create_nanomamba_nc_large(n_classes=12, use_nasg=True).to(device)
ckpt_path = './checkpoints_full/NanoMamba-NC-Large-NASG/best.pt'
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded NC-SSM-Large-NASG: val_acc={ckpt.get("val_acc", "N/A")}')
else:
    raise FileNotFoundError(
        f'{ckpt_path} not found!\n'
        '→ Run Step 12 first to train NASG with SNR Teacher-Student')

model.eval()

# ============================================================
# 11b: Load BC-ResNet-1 (CNN baseline for comparison, optional)
# ============================================================
cnn_model = None
cnn_path = './checkpoints_full/BC-ResNet-1/best.pt'
if os.path.exists(cnn_path):
    cnn_model = BCResNet(n_classes=12, scale=1).to(device)
    cnn_ckpt = torch.load(cnn_path, map_location=device)
    cnn_model.load_state_dict(cnn_ckpt['model_state_dict'])
    print(f'Loaded BC-ResNet-1: val_acc={cnn_ckpt.get("val_acc", "N/A")}')
    cnn_model.eval()
else:
    print(f'BC-ResNet-1 not found — SSM-only diagnostic (run Step 2 for comparison)')

mel_fb = _create_mel_fb_tensor()

# ============================================================
# 11c: Load validation dataset
# ============================================================
val_dataset = SpeechCommandsDataset('./data/speech_commands_v0.02', split='validation')
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)

# ============================================================
# 11d: Run full diagnostic
# ============================================================
diagnose_noise_propagation(
    model, val_loader, device,
    cnn_model=cnn_model,
    mel_fb=mel_fb,
    noise_types=('white', 'pink', 'factory', 'babble'),
    snr_dbs=(-15, -10, -5, 0, 5, 10),
    max_batches=5
)

# ============================================================
# 11e: Detailed per-block analysis
# ============================================================
print('\n' + '='*70)
print('  PER-BLOCK PARAMETER CHECK (SNR Teacher-Student)')
print('='*70)
for i, block in enumerate(model.blocks):
    ssm = block.sa_ssm
    if hasattr(ssm, 'nasg_scale'):
        print(f'\n  Block {i}:')
        print(f'    nasg_scale = {ssm.nasg_scale.item():.2f} (init=5.0)')
        print(f'    nasg_bias  = {ssm.nasg_bias.item():.2f} (init=0.0)')
        if hasattr(ssm, 'state_mask_scale'):
            print(f'    state_mask_scale = {ssm.state_mask_scale.item():.2f} (init=3.0)')
            print(f'    state_mask_bias  = {ssm.state_mask_bias.item():.2f} (init=0.0)')
        if hasattr(ssm, '_last_nasg_w'):
            g = ssm._last_nasg_w
            print(f'    Last NASG gate: min={g.min().item():.6f} max={g.max().item():.6f} mean={g.mean().item():.6f}')
        if hasattr(ssm, '_last_state_gate'):
            sg = ssm._last_state_gate
            print(f'    Last state_gate: {" ".join(f"{v:.3f}" for v in sg.mean(0))}')

# SF-to-SNR bridge params
if hasattr(model, 'sf_to_snr_scale'):
    print(f'\n  SF-to-SNR Bridge (Detached SF Supervision):')
    print(f'    sf_to_snr_scale = {model.sf_to_snr_scale.item():.4f} (init=-5.0)')
    print(f'    sf_to_snr_bias  = {model.sf_to_snr_bias.item():.4f} (init=3.4)')

print('\n' + '='*70)
print('  INTERPRETATION GUIDE (SNR Teacher-Student)')
print('='*70)
print('  1. NASG gate g (with snr_hint = tanh(SNR_dB/10)):')
print('     g ~ 0.01 at -15dB = GOOD (pure LTI, O(sigma^2))')
print('     g > 0.05 at -15dB = BAD (selectivity leaking)')
print()
print('  2. Amp.Ratio scaling:')
print('     -15dB to -5dB should decrease ~10x = O(sigma^2) confirmed')
print()
print('  3. ||h||^2 per state:')
print('     Higher states (6,7) should be ~0 at -15dB = input gating works')
print()
print('  4. N_eff:')
print('     -15dB: ~0.2-1.0 (from 8) = effective dimension reduction')
print('     -15dB: ~8 (no reduction) = state masking too weak')
print()
print('  5. SF-to-SNR Bridge:')
print('     sf_to_snr_scale should have moved from -5.0 (gradient received)')
print('     sf_to_snr_bias should have moved from 3.4 (gradient received)')
print('='*70)

## Step 12: SNR Teacher-Student Fix — NASG Gate 근본 수정

### 문제 진단 (Step 11 결과)
NASG gate가 **SNR 레벨에 불감** — White -15dB에서 g=0.857 (기대: 0.001)

**근본 원인**: SNREstimator가 첫 5프레임을 "순수 노이즈"로 가정하지만,
Speech Commands에서는 모든 프레임에 speech+noise 혼합 → SNR 추정이 일정

### 해결: SNR Teacher-Student with Detached SF Bridge
| 단계 | SNR 소스 | 정확도 |
|------|---------|--------|
| **학습** | True SNR_dB (oracle) → `snr_hint = tanh(SNR/10)` | 완벽 |
| **추론** | Spectral Flatness → 학습된 매핑 | 양호 |

**핵심: 추정 분산 차단**
- `mel.detach()`: SF 추정 오차가 feature extractor로 역전파 차단
- Auxiliary MSE loss (weight=0.5): sf_to_snr_scale/bias만 학습 (2 파라미터)
- NASG gate는 학습 시 항상 oracle 사용 → 추정 분산 전파 없음

### 수정 후 NASG gate 동작
```
-15dB: g = σ(5×(-0.905)) = 0.011  → LTI mode (O(σ²))
 -5dB: g = σ(5×(-0.462)) = 0.091  → 약한 selectivity
  0dB: g = σ(5×(0.000))  = 0.500  → 절반
  5dB: g = σ(5×(0.462))  = 0.909  → 강한 selectivity
Clean: g = σ(5×(0.987))  = 0.993  → 전체 selectivity
```

| 파라미터 | 이전 | 수정 후 |
|---------|------|---------|
| nasg_scale | 20.0 | **5.0** |
| nasg_bias | -5.0 | **0.0** |
| state_mask_scale | 8.0 | **3.0** |
| state_mask_bias | -2.0 | **0.0** |
| sf_to_snr_scale | (없음) | **-5.0** (fitted to SF distribution) |
| sf_to_snr_bias | (없음) | **3.4** (fitted to SF distribution) |
| 총 파라미터 | 10,199 | **10,201** (+2) |

In [ ]:
%%time
import os, shutil, torch

# ============================================================
# 12-PREREQ: Self-contained setup (works without Step 1)
# ============================================================
REPO_DIR = '/content/NC-SSM-TASLP'
LOCAL_CKPT = os.path.join(REPO_DIR, 'checkpoints_full')
DRIVE_CKPT = '/content/drive/MyDrive/NC-SSM-TASLP/checkpoints_full'
DRIVE_OLD  = '/content/drive/MyDrive/NanoMamba/checkpoints_full'
DATA_DIR   = os.path.join(REPO_DIR, 'data')

# --- Clone or pull ---
if os.path.exists(REPO_DIR):
    print('[PREREQ] Repo exists, pulling latest...')
    os.system(f'cd {REPO_DIR} && git pull origin master')
else:
    print('[PREREQ] Cloning repo...')
    os.system(f'git clone https://github.com/DrJinHoChoi/NC-SSM-TASLP.git {REPO_DIR}')
os.chdir(REPO_DIR)

# --- Download GSC V2 if missing ---
GSC_DIR = os.path.join(DATA_DIR, 'speech_commands_v0.02')
if not os.path.exists(GSC_DIR):
    print('[PREREQ] Downloading GSC V2...')
    os.system(f'mkdir -p {DATA_DIR} && cd {DATA_DIR} && '
              'wget -q http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz && '
              'mkdir -p speech_commands_v0.02 && '
              'tar -xzf speech_commands_v0.02.tar.gz -C speech_commands_v0.02 && '
              'rm speech_commands_v0.02.tar.gz')
    print('[PREREQ] GSC V2 downloaded')
else:
    print(f'[PREREQ] GSC V2 already at {GSC_DIR}')

# --- Restore baseline checkpoints from Drive (if available) ---
os.makedirs(LOCAL_CKPT, exist_ok=True)
for model_name in ['NanoMamba-NC-Large', 'BC-ResNet-1']:
    local_best = os.path.join(LOCAL_CKPT, model_name, 'best.pt')
    if os.path.exists(local_best):
        continue
    for drv in [DRIVE_CKPT, DRIVE_OLD]:
        src = os.path.join(drv, model_name, 'best.pt')
        if os.path.exists(src):
            os.makedirs(os.path.join(LOCAL_CKPT, model_name), exist_ok=True)
            shutil.copy2(src, local_best)
            print(f'[PREREQ] Restored {model_name} from Drive')
            break

# --- save_to_drive helper ---
def save_to_drive():
    """Sync local checkpoints to Drive for persistence."""
    os.makedirs(DRIVE_CKPT, exist_ok=True)
    for d in os.listdir(LOCAL_CKPT):
        src = os.path.join(LOCAL_CKPT, d, 'best.pt')
        if os.path.exists(src):
            dst_dir = os.path.join(DRIVE_CKPT, d)
            os.makedirs(dst_dir, exist_ok=True)
            shutil.copy2(src, os.path.join(dst_dir, 'best.pt'))
    print(f'✓ Checkpoints synced to Drive ({DRIVE_CKPT})')

print('[PREREQ] Setup complete\n')

# ============================================================
# 12a: Delete old NASG checkpoint (incompatible)
# ============================================================
print('='*70)
print('12a: Preparing for SNR Teacher-Student training')
print('='*70)

old_nasg = os.path.join(LOCAL_CKPT, 'NanoMamba-NC-Large-NASG')
if os.path.exists(old_nasg):
    shutil.rmtree(old_nasg)
    print('  Deleted old NASG checkpoint (NASG scale/bias changed)')
drive_nasg = os.path.join(DRIVE_CKPT, 'NanoMamba-NC-Large-NASG')
if os.path.exists(drive_nasg):
    shutil.rmtree(drive_nasg)

# ============================================================
# 12b: Train NASG with SNR Teacher-Student (10,201 params)
#      Training: snr_hint = tanh(SNR_dB/10) passed as oracle
#      Detached SF Bridge: mel.detach() -> SF -> auxiliary MSE loss
#      NASG gate correctly responds to SNR LEVEL
# ============================================================
print('\n' + '='*70)
print('12b: Training NASG with SNR Teacher-Student + Detached SF Bridge')
print('    snr_hint = tanh(SNR_dB/10) -> NASG gate responds to SNR level')
print('    nasg_scale=5.0, nasg_bias=0.0 (re-tuned for [-1,1] input)')
print('    sf_to_snr_scale=-5.0, sf_to_snr_bias=3.4 (fitted to SF distribution)')
print('    Auxiliary MSE loss (w=0.5): trains SF bridge with mel.detach()')
print('    Total: 10,201 params (+2 for SF->SNR mapping)')
print('='*70)

!python train_colab.py \
    --models NanoMamba-NC-Large-NASG \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --noise_aug --noise_curriculum_v2 --low_snr_boost \
    --calibrate

save_to_drive()
print('\n  ✓ NASG Teacher-Student saved to Drive')

# ============================================================
# 12c: Full eval — NASG Teacher-Student (calibrated, SF inference)
#      At inference: snr_hint=None -> learned SF bridge
#      Only evaluate models whose checkpoints exist
# ============================================================
print('\n' + '='*70)
print('12c: NASG Teacher-Student full noise evaluation')
print('     Inference: SF-based SNR estimation (no oracle)')
print('='*70)

# Build model list: always NASG, add baselines only if checkpoints exist
eval_models = ['NanoMamba-NC-Large-NASG']
for baseline in ['NanoMamba-NC-Large', 'BC-ResNet-1']:
    bp = os.path.join(LOCAL_CKPT, baseline, 'best.pt')
    if os.path.exists(bp):
        eval_models.append(baseline)
        print(f'  ✓ {baseline} checkpoint found — will compare')
    else:
        print(f'  ✗ {baseline} not found — skipping (run Step 2 to train)')

eval_str = ','.join(eval_models)
print(f'  Evaluating: {eval_str}\n')

!python train_colab.py \
    --models {eval_str} \
    --eval_only \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15 \
    --calibrate --calibrate_continuous

# ============================================================
# 12d: NASG Teacher-Student + SS v2 + Bypass (calibrated)
# ============================================================
print('\n' + '='*70)
print('12d: NASG Teacher-Student + SS v2 + Bypass')
print('='*70)

!python train_colab.py \
    --models NanoMamba-NC-Large-NASG \
    --eval_only \
    --use_enhancer --enhancer_bypass \
    --ss_version v2 --bypass_version v2 \
    --bypass_threshold -2.0 --bypass_scale 3.0 --sf_range 2.0 \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15 \
    --calibrate --calibrate_continuous

print('\n' + '='*70)
print('Step 12 RESULTS:')
print('  Before fix (NASG v2):        g=0.857 at -15dB, White -15dB=33.5%')
print('  After fix (Teacher-Student):  g=0.011 at -15dB, White -15dB=??? (target 48-55%)')
print('  + SS v2:                      White -15dB=??? (target 55-62%)')
print('  BC-ResNet-1 baseline:         White -15dB=58.6%')
print('='*70)
print('\n→ Next: Run Step 11 to verify O(σ²) noise propagation')

[PREREQ] Repo exists, pulling latest...
[PREREQ] GSC V2 already at /content/NC-SSM-TASLP/data/speech_commands_v0.02
[PREREQ] Setup complete

12a: Preparing for SNR Teacher-Student training
  Deleted old NASG checkpoint (NASG scale/bias changed)

12b: Training NASG with SNR Teacher-Student + Detached SF Bridge
    snr_hint = tanh(SNR_dB/10) -> NASG gate responds to SNR level
    nasg_scale=5.0, nasg_bias=0.0 (re-tuned for [-1,1] input)
    sf_to_snr_scale=-5.0, sf_to_snr_bias=3.4 (fitted to SF distribution)
    Auxiliary MSE loss (w=0.5): trains SF bridge with mel.detach()
    Total: 10,201 params (+2 for SF->SNR mapping)
  [OK] nanomamba.py loaded successfully

  NanoMamba Training — Structural Noise Robustness
  Device: cuda
  Models: NanoMamba-NC-Large-NASG
  Epochs: 30, LR: 0.003, Batch: 128
  Time: 2026-03-15 02:17:06

  Loading Google Speech Commands V2...
  Loading training split via torchaudio...
  [training] 86843 samples, 12 classes
    Per-class: {'down': 3134, 'go': 3106, 'l